# Runtime Isolation and Sandboxing for MCP Servers

**Goal:** admit a pinned least-privilege workload, execute it in a real bounded subprocess, attack the boundary, and translate the invariant set to a production sandbox without confusing policy with enforcement.

This notebook uses no external credentials, container daemon, or cloud account.

## 1. Load the executable lab

The notebook imports the same implementation covered by the focused test suite.

In [ ]:
import json
import runpy
import tempfile
from pathlib import Path

lab = runpy.run_path(Path("lab.py"))
admission, runtime, service = lab["default_environment"]()
print("Loaded Course 09 bounded runtime")

## 2. Locate the real trust boundaries

MCP stdio launches a subprocess; it does not sandbox that process. Consent, artifact admission, application authorization, runtime enforcement, and output verification are separate controls.

The portable lab applies only process-level controls. It reports stronger production controls as not enforced rather than turning configuration names into security evidence.

In [ ]:
{
    "worker_fingerprint": admission.worker_fingerprint(admission.policy.worker_digest),
    "locally_enforced": lab["LOCAL_ENFORCED_CONTROLS"],
    "not_claimed": lab["LOCAL_NOT_ENFORCED"],
}

## 3. Keep runtime authority out of the MCP schema

The model may select an opaque file ID. It may not choose a command, path, tenant, image, environment variable, destination, resource limit, or runtime class.

In [ ]:
server = lab["build_mcp_server"](service)

async def list_schema():
    async with lab["Client"](server) as client:
        tools = await client.list_tools()
        return tools.tools[0].input_schema

tool_schema = await list_schema()
tool_schema

## 4. Execute a real bounded child process

Trusted code resolves the record, copies it into a fresh input tree, launches an exact Python argument vector with no shell, applies available POSIX limits, uses file-backed bounded output, validates the exit state, and parses JSON.

In [ ]:
request = lab["request_for"](request_id="run-notebook-valid")
record = admission.require(request)
result = runtime.execute(request, record)
result

### Interpret the environment evidence

The child sees an explicit safe environment. macOS may inject `__CF_USER_TEXT_ENCODING`; the test suite allows only that documented platform key. Secret values and the parent environment are never printed.

In [ ]:
expected_keys = set(lab["SAFE_ENVIRONMENT"]) | set(lab["PLATFORM_ENVIRONMENT_KEYS"])
{
    "observed": result["environment_keys"],
    "expected_only": set(result["environment_keys"]) == expected_keys,
}

## 5. Attack admission before launch

These cases test unknown data, cross-tenant access, ambient-secret requests, and network requests. A denial means no worker started.

In [ ]:
negative_requests = [
    ("unknown file", lab["request_for"](request_id="run-nb-unknown", file_id="file-unknown")),
    ("cross tenant", lab["request_for"](request_id="run-nb-tenant", file_id="file-other-tenant")),
    ("secret environment", lab["request_for"](request_id="run-nb-env", env_keys=frozenset({"LANG", "API_TOKEN"}))),
    ("network request", lab["request_for"](request_id="run-nb-network", network_destination="https://evil.example")),
]
admission_results = []
runtime_events_before = len(runtime.events)
for name, attack in negative_requests:
    decision = admission.evaluate(attack)
    admission_results.append({"case": name, "decision": decision.reason_code})
{
    "results": admission_results,
    "workers_started": len(runtime.events) - runtime_events_before,
    "blocked_attempt_rate": sum(item["decision"] != "ADMISSION_ALLOWED" for item in admission_results) / len(admission_results),
}

## 6. Prove canonical path resolution rejects a symlink escape

Application resolution is useful defense in depth. Arbitrary compromised native code still requires an OS or VM filesystem boundary.

In [ ]:
with tempfile.TemporaryDirectory() as temporary:
    base = Path(temporary)
    root = base / "input"
    root.mkdir()
    outside = base / "secret.txt"
    outside.write_text("not for the worker")
    (root / "link.txt").symlink_to(outside)
    try:
        lab["resolve_beneath"](root, root / "link.txt")
        symlink_result = "UNEXPECTED_ALLOW"
    except lab["SandboxDenied"] as exc:
        symlink_result = exc.code
symlink_result

## 7. Inject runtime failures

The fault selector is trusted test instrumentation and is not exposed by the MCP tool. A hung child must reach `terminated`, while oversized output must hit the file-size boundary. Neither may become successful tool data.

In [ ]:
failure_results = []
for index, fault in enumerate(("hang", "oversize", "invalid-json", "fail"), start=1):
    fault_request = lab["request_for"](request_id=f"run-nb-fault-{index}")
    fault_record = admission.require(fault_request)
    try:
        runtime.execute(fault_request, fault_record, trusted_fault=fault)
        failure_results.append({"fault": fault, "outcome": "UNEXPECTED_SUCCESS"})
    except lab["SandboxExecutionFailed"] as exc:
        failure_results.append({"fault": fault, "outcome": exc.code})
failure_results

## 8. Inspect redacted execution evidence

Evidence names the control layer and terminal state. It does not contain document text, environment values, worker source, or hidden reasoning.

In [ ]:
[event.__dict__ for event in runtime.events[-5:]]

## 9. Map invariants to a production sandbox profile

The profile requires a digest-pinned image, sandbox runtime class, non-root identity, read-only root, no privilege escalation, no privileged mode, all capabilities dropped, RuntimeDefault seccomp/AppArmor, no host namespaces or service-account token, resource limits, and network default deny.

In [ ]:
production_profile = lab["ProductionIsolationProfile"]()
bundle = lab["restricted_kubernetes_bundle"](production_profile)
{
    "runtime_class": bundle["pod"]["spec"]["runtimeClassName"],
    "image": bundle["pod"]["spec"]["containers"][0]["image"],
    "security_context": bundle["pod"]["spec"]["containers"][0]["securityContext"],
    "limits": bundle["pod"]["spec"]["containers"][0]["resources"]["limits"],
    "network_policy": bundle["network_policy"]["spec"],
}

### Configuration is not deployment proof

Kubernetes has no standard API definition for a sandboxed Pod. Verify that Pod Security Admission, RuntimeClass scheduling, seccomp/AppArmor, CNI NetworkPolicy, PID limiting, image verification, and node hardening are active on the deployed workload.

In [ ]:
bundle["operator_notes"]

## 10. Choose the boundary by threat model

| Workload | Starting tier | Why |
|---|---|---|
| reviewed deterministic helper | bounded process or restricted container | low code uncertainty, still needs resource and secret controls |
| third-party MCP over private data | sandboxed container | stronger kernel mediation with container compatibility |
| narrow portable function | Wasm/WASI | capability-oriented host imports and preopened resources |
| arbitrary multi-tenant native code | microVM or dedicated worker | independent guest kernel or hardware boundary |

Benchmark compatibility and operational cost, but do not infer security strength from startup latency.

## 11. Evaluate the right outcomes

A blocked request is not an escape. An escape is not merely a suspicious log line. Use labelled populations and keep blocked attempts, policy bypasses, sandbox escapes, forbidden effects, and false denials separate.

In [ ]:
expected_denials = len(admission_results) + len(failure_results) + 1  # symlink
correct_denials = sum(item["decision"] != "ADMISSION_ALLOWED" for item in admission_results)
correct_denials += sum(item["outcome"] != "UNEXPECTED_SUCCESS" for item in failure_results)
correct_denials += symlink_result != "UNEXPECTED_ALLOW"
{
    "population": expected_denials,
    "blocked_attempt_rate": correct_denials / expected_denials,
    "fixture_forbidden_effect_rate": 0 / expected_denials,
    "direction": "blocked rate higher is better; forbidden-effect rate lower is better",
}

## 12. Run the complete real-MCP scenario

The isolated scenario uses a fresh environment, calls the actual MCP tool, triggers policy and runtime denials, and returns a claim ledger.

In [ ]:
evidence = await lab["run_scenario"]()
evidence

## Production upgrades

- Select a standard container, gVisor, Kata, Firecracker, Wasmtime, or dedicated worker from the real tenant and host threat model.
- Verify artifact provenance before admission and runtime evidence after scheduling.
- Add OS-enforced filesystem and network isolation, cgroups, seccomp/LSM, short-lived workload identity, cleanup, patching, and incident containment.
- Test cancellation, process-tree cleanup, scratch deletion, credential revocation, and late-result suppression.

## Reflection

1. Which controls did this host actually enforce, and which exist only in the production profile?
2. Why does an allowed MCP tool call still need a runtime boundary?
3. What evidence would prove a Kubernetes workload really used gVisor?
4. Which change in your own threat model would move a workload from a container to a microVM?